#  Global Development Analysis using Gapminder Dataset
### Interactive Visualizations with Plotly & Dash

**Author:** Om Patil 
**Dataset:** Gapminder Dataset (CSV)  
**Tools:** Python, Pandas, NumPy, Plotly, Dash

---

## 1. Introduction

### What is the Gapminder Dataset?

The **Gapminder dataset** is a collection of global development indicators compiled by the Gapminder Foundation — a non-profit organization that promotes a fact-based worldview. It tracks socioeconomic and health indicators across countries and continents over time.

### Why is it Useful for Global Development Analysis?

- It provides **multi-decade trends** for countries across every continent
- It combines **health, economic, and environmental** indicators in one dataset
- It enables **cross-country and cross-continent comparisons**
- It is widely used for **storytelling with data** (popularized by Hans Rosling's famous TED Talks)

### Dataset Column Descriptions

| Column | Description |
|---|---|
| **country** | Name of the country |
| **continent** | Continent the country belongs to |
| **year** | Year of observation |
| **life_exp** | Average life expectancy at birth (years) |
| **hdi_index** | Human Development Index (0–1 scale, higher = better) |
| **co2_consump** | CO₂ consumption per capita (metric tons) |
| **gdp** | Gross Domestic Product per capita (USD) |
| **services** | Services sector as % of GDP |

### What Insights Do We Want to Discover?

- How does **GDP relate to life expectancy** across countries?
- Which continents lead in **human development (HDI)**?
- How has **life expectancy changed** over the years?
- Which countries produce the most **CO₂** and how does it relate to wealth?
- What are global **development trends** from 1998 to 2007?

---

## 2. Import Libraries

We import all required libraries. **Plotly Express** provides high-level, easy-to-use chart functions, while **Plotly Graph Objects** gives fine-grained control. **Dash** is used to build the interactive dashboard.

In [47]:
import sys
print(sys.executable)

c:\Users\91737\miniconda3\python.exe


In [48]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"Plotly version : {plotly.__version__}")
print(f"Dash version   : {dash.__version__}")
print("Libraries imported successfully! ")

NumPy version  : 2.3.5
Pandas version : 2.3.3
Plotly version : 6.5.0
Dash version   : 4.0.0
Libraries imported successfully! 


---

## 3. Load the Dataset

We load the Gapminder dataset from the provided CSV file using Pandas. We then inspect its structure, columns, data types, and the unique continents present.

In [49]:
# Load the dataset
df = pd.read_csv('D:\Data _Visualization(M512A)\Week_three\gapminder_data_graphs.csv')

print("=" * 55)
print("First 5 Rows:")
print("=" * 55)
print(df.head())

print(f"\nDataset Shape     : {df.shape}")
print(f"Columns           : {list(df.columns)}")
print(f"Year Range        : {df['year'].min()} — {df['year'].max()}")
print(f"Unique Continents : {df['continent'].unique().tolist()}")

print("\n" + "=" * 55)
print("Data Types:")
print("=" * 55)
print(df.dtypes)

First 5 Rows:
       country continent  year  life_exp  hdi_index  co2_consump    gdp  \
0  Afghanistan      Asia  1998      53.3      0.344       0.0522    NaN   
1  Afghanistan      Asia  1999      54.7      0.348       0.0402    NaN   
2  Afghanistan      Asia  2000      54.7      0.350       0.0370    NaN   
3  Afghanistan      Asia  2001      54.8      0.353       0.0376    NaN   
4  Afghanistan      Asia  2002      55.5      0.384       0.0471  333.0   

   services  
0      24.4  
1      24.6  
2      24.7  
3      24.7  
4      25.6  

Dataset Shape     : (3675, 8)
Columns           : ['country', 'continent', 'year', 'life_exp', 'hdi_index', 'co2_consump', 'gdp', 'services']
Year Range        : 1998 — 2018
Unique Continents : ['Asia', 'Europe', 'Africa', 'South America', 'Oceania', 'North America']

Data Types:
country         object
continent       object
year             int64
life_exp       float64
hdi_index      float64
co2_consump    float64
gdp            float64
services

---

## 4. Data Cleaning

Before visualization, we must clean the data to ensure accuracy:
- **Missing values** in `gdp`, `hdi_index`, and `co2_consump` are filled with column medians
- **Duplicate rows** could distort aggregations and are removed if found
- **Summary statistics** give an overview of each numerical column's range and distribution

In [50]:
print("Missing Values Before Cleaning:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

# Fill missing values with column medians
for col in ['gdp', 'hdi_index', 'co2_consump']:
    df[col].fillna(df[col].median(), inplace=True)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

print("\n" + "=" * 55)
print("Summary Statistics:")
print("=" * 55)
print(df.describe().round(2))

Missing Values Before Cleaning:
country          0
continent        0
year             0
life_exp         0
hdi_index      112
co2_consump      4
gdp             42
services         0
dtype: int64

Duplicate Rows: 0

Missing Values After Cleaning:
country        0
continent      0
year           0
life_exp       0
hdi_index      0
co2_consump    0
gdp            0
services       0
dtype: int64

Summary Statistics:
          year  life_exp  hdi_index  co2_consump        gdp  services
count  3675.00   3675.00    3675.00      3675.00    3675.00   3675.00
mean   2008.00     69.85       0.68         4.71   11878.21     51.25
std       6.06      8.89       0.16         6.56   17027.35     18.31
min    1998.00     32.50       0.26         0.02     238.00      5.59
25%    2003.00     63.90       0.54         0.56    1500.00     37.60
50%    2008.00     71.70       0.70         2.25    4280.00     52.90
75%    2013.00     76.40       0.80         6.61   13400.00     65.70
max    2018.00     84.

---

## 5. Exploratory Data Analysis (EDA)

Our analysis focuses on three main goals:
1. **Compare development indicators** across continents — Which regions are most developed?
2. **Analyze trends over time** — How have health and economic indicators evolved?
3. **Explore relationships** — Does higher GDP lead to longer life expectancy? Is CO₂ consumption tied to wealth?

In [51]:
# Prepare commonly used subsets
df_2007 = df[df['year'] == 2007].copy()

# Ensure no NaN in columns used for bubble size
df_2007['hdi_index'] = df_2007['hdi_index'].fillna(df_2007['hdi_index'].median())
df_2007['life_exp']  = df_2007['life_exp'].fillna(df_2007['life_exp'].median())

print("Countries in 2007 dataset:", df_2007['country'].nunique())
print("\nAverage GDP by Continent (2007):")
print(df_2007.groupby('continent')['gdp'].mean().sort_values(ascending=False).round(2))

Countries in 2007 dataset: 175

Average GDP by Continent (2007):
continent
Europe           27526.92
Oceania          13552.50
North America    13003.68
Asia             11734.77
South America     6989.09
Africa            2241.45
Name: gdp, dtype: float64


---

## Visualization 1 — Histogram: Population Distribution in 2007

This histogram shows the **distribution of GDP per capita** across all countries in 2007. Most countries are clustered at lower GDP values, confirming the wide global inequality in wealth.

In [52]:
fig1 = px.histogram(
    df_2007,
    x='gdp',
    color='continent',
    nbins=40,
    title='GDP per Capita Distribution by Continent (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'count': 'Number of Countries'},
    template='plotly_white',
    barmode='overlay',
    opacity=0.75
)
fig1.update_layout(
    title_font_size=16,
    xaxis_title='GDP per Capita (USD)',
    yaxis_title='Number of Countries',
    legend_title='Continent'
)
fig1.show()

---

## Visualization 2 — Line Plot: Average Life Expectancy Over Time

This line plot tracks the **average life expectancy per continent** from 1998 to 2007. It clearly shows which continents have improved health outcomes the most and which lag behind.

In [53]:
life_exp_trend = df.groupby(['year', 'continent'])['life_exp'].mean().reset_index()

fig2 = px.line(
    life_exp_trend,
    x='year',
    y='life_exp',
    color='continent',
    markers=True,
    title='Average Life Expectancy by Continent Over Time',
    labels={'life_exp': 'Avg Life Expectancy (years)', 'year': 'Year'},
    template='plotly_white'
)
fig2.update_layout(
    title_font_size=16,
    legend_title='Continent',
    xaxis=dict(dtick=1)
)
fig2.show()

---

## Visualization 3 — Scatter Plot: GDP vs Life Expectancy (2007)

This scatter plot compares **GDP per capita vs life expectancy** for all countries in 2007. Points are colored by continent and sized by HDI index. This is the classic Gapminder visualization — it powerfully shows the relationship between wealth and health.

In [54]:
fig3 = px.scatter(
    df_2007,
    x='gdp',
    y='life_exp',
    color='continent',
    size='hdi_index',
    hover_name='country',
    hover_data={'gdp': True, 'life_exp': True, 'hdi_index': True},
    title='GDP per Capita vs Life Expectancy (2007)',
    labels={
        'gdp': 'GDP per Capita (USD)',
        'life_exp': 'Life Expectancy (years)',
        'hdi_index': 'HDI Index'
    },
    template='plotly_white',
    size_max=25
)
fig3.update_layout(title_font_size=16, legend_title='Continent')
fig3.show()

---

## Visualization 4 — Bar Plot: Top 20 Countries by GDP (2007)

This horizontal bar plot ranks the **top 20 countries by GDP per capita** in 2007. Bars are colored by continent, making it easy to see which regions dominate the highest income brackets.

In [55]:
top20_gdp = df_2007.nlargest(20, 'gdp').sort_values('gdp')

fig4 = px.bar(
    top20_gdp,
    x='gdp',
    y='country',
    color='continent',
    orientation='h',
    title='Top 20 Countries by GDP per Capita (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'country': 'Country'},
    template='plotly_white',
    hover_data=['life_exp', 'hdi_index']
)
fig4.update_layout(title_font_size=16, legend_title='Continent', height=550)
fig4.show()

---

## Visualization 5 — World Map: Life Expectancy by Country (2007)

An interactive **choropleth world map** showing life expectancy across all countries in 2007. Darker colors indicate higher life expectancy. Hover over any country to see detailed information.

In [56]:
fig5 = px.choropleth(
    df_2007,
    locations='country',
    locationmode='country names',
    color='life_exp',
    hover_name='country',
    hover_data={'gdp': True, 'hdi_index': True, 'co2_consump': True},
    color_continuous_scale='RdYlGn',
    title='World Map — Life Expectancy by Country (2007)',
    labels={'life_exp': 'Life Expectancy (years)'},
    template='plotly_white'
)
fig5.update_layout(
    title_font_size=16,
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar=dict(title='Life Exp (yrs)')
)
fig5.show()

---

## Visualization 6 — Box Plot: Life Expectancy Distribution by Continent

Box plots reveal the **spread, median, and outliers** of life expectancy within each continent. This helps compare not just averages but also inequality within continents.

In [57]:
fig6 = px.box(
    df_2007,
    x='continent',
    y='life_exp',
    color='continent',
    points='all',
    hover_name='country',
    title='Life Expectancy Distribution by Continent (2007)',
    labels={'life_exp': 'Life Expectancy (years)', 'continent': 'Continent'},
    template='plotly_white'
)
fig6.update_layout(title_font_size=16, showlegend=False)
fig6.show()

---

## Visualization 7 — Box Plot: GDP Distribution by Continent

This box plot shows the **GDP per capita distribution across continents** in 2007. Europe and North America show significantly higher medians, while Africa has the lowest — with a few high-income outliers.

In [58]:
fig7 = px.box(
    df_2007,
    x='continent',
    y='gdp',
    color='continent',
    points='outliers',
    hover_name='country',
    title='GDP per Capita Distribution by Continent (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'continent': 'Continent'},
    template='plotly_white'
)
fig7.update_layout(title_font_size=16, showlegend=False)
fig7.show()

---

## Visualization 8 — Scatter Plot: CO₂ Consumption vs GDP

This scatter plot explores the **relationship between CO₂ consumption and GDP per capita**. Wealthier countries tend to consume more energy and emit more CO₂ — but some countries are more efficient than others.

In [59]:
fig8 = px.scatter(
    df_2007,
    x='gdp',
    y='co2_consump',
    color='continent',
    hover_name='country',
    size='life_exp',
    title='CO₂ Consumption vs GDP per Capita (2007)',
    labels={
        'gdp': 'GDP per Capita (USD)',
        'co2_consump': 'CO₂ Consumption per Capita (metric tons)'
    },
    template='plotly_white',
    size_max=20
)
fig8.update_layout(title_font_size=16, legend_title='Continent')
fig8.show()

---

## Visualization 9 — Scatter Plot: HDI vs Life Expectancy

The **Human Development Index (HDI)** combines education, income, and health. This scatter plot shows how closely HDI aligns with life expectancy — a strong positive correlation is expected.

In [60]:
fig9 = px.scatter(
    df_2007,
    x='hdi_index',
    y='life_exp',
    color='continent',
    hover_name='country',
    trendline='ols',
    title='HDI Index vs Life Expectancy (2007)',
    labels={
        'hdi_index': 'Human Development Index (HDI)',
        'life_exp': 'Life Expectancy (years)'
    },
    template='plotly_white'
)
fig9.update_layout(title_font_size=16, legend_title='Continent')
fig9.show()

---

## Visualization 10 — Bar Plot: Services Sector by Continent

The **services sector as % of GDP** reflects how economically advanced a country is. Developed nations tend to have a higher share of services (finance, tech, healthcare) compared to manufacturing or agriculture.

In [61]:
services_continent = df_2007.groupby('continent')['services'].mean().reset_index()
services_continent = services_continent.sort_values('services', ascending=False)

fig10 = px.bar(
    services_continent,
    x='continent',
    y='services',
    color='continent',
    title='Average Services Sector (% of GDP) by Continent (2007)',
    labels={'services': 'Services (% of GDP)', 'continent': 'Continent'},
    template='plotly_white',
    text_auto='.1f'
)
fig10.update_layout(title_font_size=16, showlegend=False)
fig10.show()

---

## Visualization 11 — Line Plot: Life Expectancy Trend for Selected Countries

We select 6 representative countries and track their **life expectancy over the full time range** (1998–2007). This reveals which nations have made the fastest improvements.

In [62]:
selected_countries = ['United States', 'China', 'India', 'Brazil', 'Germany', 'Nigeria']
df_selected = df[df['country'].isin(selected_countries)]

fig11 = px.line(
    df_selected,
    x='year',
    y='life_exp',
    color='country',
    markers=True,
    title='Life Expectancy Trend for Selected Countries (1998–2007)',
    labels={'life_exp': 'Life Expectancy (years)', 'year': 'Year'},
    template='plotly_white'
)
fig11.update_layout(title_font_size=16, legend_title='Country', xaxis=dict(dtick=1))
fig11.show()

---

## Visualization 12 — Line Plot: CO₂ Consumption Trend Over Time

This plot shows the **average CO₂ consumption per capita by continent** over time. It helps identify which regions are increasing or reducing their carbon footprint.

In [63]:
co2_trend = df.groupby(['year', 'continent'])['co2_consump'].mean().reset_index()

fig12 = px.line(
    co2_trend,
    x='year',
    y='co2_consump',
    color='continent',
    markers=True,
    title='Average CO₂ Consumption per Capita by Continent (1998–2007)',
    labels={'co2_consump': 'CO₂ per Capita (metric tons)', 'year': 'Year'},
    template='plotly_white'
)
fig12.update_layout(title_font_size=16, legend_title='Continent', xaxis=dict(dtick=1))
fig12.show()

---

## Visualization 13 — Histogram: HDI Index Distribution

The **HDI (Human Development Index)** ranges from 0 to 1. This histogram shows how HDI values are distributed globally in 2007 — revealing how many countries are in low, medium, and high development categories.

In [64]:
fig13 = px.histogram(
    df_2007,
    x='hdi_index',
    color='continent',
    nbins=30,
    title='HDI Index Distribution by Continent (2007)',
    labels={'hdi_index': 'Human Development Index (HDI)'},
    template='plotly_white',
    barmode='overlay',
    opacity=0.75
)
fig13.update_layout(title_font_size=16, legend_title='Continent',
                    xaxis_title='HDI Index', yaxis_title='Count')
fig13.show()

---

## Visualization 14 — Bar Plot: Top 10 Countries by GDP (2007)

A focused look at the **wealthiest 10 countries** in 2007, ranked by GDP per capita with their continent color-coded.

In [65]:
top10 = df_2007.nlargest(10, 'gdp').sort_values('gdp', ascending=True)

fig14 = px.bar(
    top10,
    x='gdp',
    y='country',
    color='continent',
    orientation='h',
    title='Top 10 Countries by GDP per Capita (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'country': 'Country'},
    template='plotly_white',
    text_auto='.0f'
)
fig14.update_layout(title_font_size=16, legend_title='Continent', height=450)
fig14.show()

---

## Visualization 15 — Bar Plot: Continent-wise Average GDP Over Time

A grouped bar chart comparing the **average GDP per capita across all continents** for each year. This shows economic growth trends across regions from 1998 to 2007.

In [66]:
gdp_continent_year = df.groupby(['year', 'continent'])['gdp'].mean().reset_index()

fig15 = px.bar(
    gdp_continent_year,
    x='year',
    y='gdp',
    color='continent',
    barmode='group',
    title='Average GDP per Capita by Continent Over Time',
    labels={'gdp': 'Avg GDP per Capita (USD)', 'year': 'Year'},
    template='plotly_white'
)
fig15.update_layout(title_font_size=16, legend_title='Continent', xaxis=dict(dtick=1))
fig15.show()

---

## Dash Dashboard

We now build an **interactive Dash dashboard** that consolidates the four key visualizations:
1. GDP Distribution Histogram
2. Life Expectancy Trend Line Chart
3. GDP vs Life Expectancy Scatter Plot
4. World Map of Life Expectancy



In [67]:
# ── Prepare data for dashboard ──────────────────────────────────
df_2007_dash = df[df['year'] == 2007].copy()
df_2007_dash['hdi_index'] = df_2007_dash['hdi_index'].fillna(df_2007_dash['hdi_index'].median())
df_2007_dash['life_exp']  = df_2007_dash['life_exp'].fillna(df_2007_dash['life_exp'].median())

life_trend      = df.groupby(['year', 'continent'])['life_exp'].mean().reset_index()
co2_trend       = df.groupby(['year', 'continent'])['co2_consump'].mean().reset_index()
gdp_cont_year   = df.groupby(['year', 'continent'])['gdp'].mean().reset_index()
services_cont   = df_2007_dash.groupby('continent')['services'].mean().reset_index().sort_values('services', ascending=False)
selected        = ['United States', 'China', 'India', 'Brazil', 'Germany', 'Nigeria']
df_selected     = df[df['country'].isin(selected)]
top10           = df_2007_dash.nlargest(10, 'gdp').sort_values('gdp', ascending=True)

# ── Build all figures ─────────────────────────────────────────────
fig_hist = px.histogram(df_2007_dash, x='gdp', color='continent', nbins=35,
    title='1. GDP per Capita Distribution (2007)',
    labels={'gdp': 'GDP per Capita (USD)'},
    template='plotly_white', barmode='overlay', opacity=0.75)

fig_line = px.line(life_trend, x='year', y='life_exp', color='continent', markers=True,
    title='2. Avg Life Expectancy by Continent Over Time',
    labels={'life_exp': 'Life Expectancy (yrs)', 'year': 'Year'},
    template='plotly_white')

fig_scatter = px.scatter(df_2007_dash, x='gdp', y='life_exp', color='continent',
    hover_name='country', size='hdi_index', size_max=20,
    title='3. GDP vs Life Expectancy (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'life_exp': 'Life Expectancy (yrs)'},
    template='plotly_white')

fig_bar_top20 = px.bar(df_2007_dash.nlargest(20, 'gdp').sort_values('gdp'),
    x='gdp', y='country', color='continent', orientation='h',
    title='4. Top 20 Countries by GDP per Capita (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'country': 'Country'},
    template='plotly_white')

fig_map = px.choropleth(df_2007_dash, locations='country', locationmode='country names',
    color='life_exp', hover_name='country', color_continuous_scale='RdYlGn',
    title='5. World Map — Life Expectancy (2007)',
    labels={'life_exp': 'Life Exp (yrs)'}, template='plotly_white')
fig_map.update_layout(geo=dict(showframe=False))

fig_box_life = px.box(df_2007_dash, x='continent', y='life_exp', color='continent',
    points='all', hover_name='country',
    title='6. Life Expectancy Distribution by Continent (2007)',
    labels={'life_exp': 'Life Expectancy (years)'}, template='plotly_white')

fig_box_gdp = px.box(df_2007_dash, x='continent', y='gdp', color='continent',
    points='outliers', hover_name='country',
    title='7. GDP Distribution by Continent (2007)',
    labels={'gdp': 'GDP per Capita (USD)'}, template='plotly_white')

fig_co2_gdp = px.scatter(df_2007_dash, x='gdp', y='co2_consump', color='continent',
    hover_name='country', size='life_exp', size_max=20,
    title='8. CO₂ Consumption vs GDP per Capita (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'co2_consump': 'CO₂ per Capita (metric tons)'},
    template='plotly_white')

fig_hdi = px.scatter(df_2007_dash, x='hdi_index', y='life_exp', color='continent',
    hover_name='country', trendline='ols',
    title='9. HDI Index vs Life Expectancy (2007)',
    labels={'hdi_index': 'HDI Index', 'life_exp': 'Life Expectancy (years)'},
    template='plotly_white')

fig_services = px.bar(services_cont, x='continent', y='services', color='continent',
    title='10. Avg Services Sector (% of GDP) by Continent (2007)',
    labels={'services': 'Services (% of GDP)'}, template='plotly_white', text_auto='.1f')

fig_country_trend = px.line(df_selected, x='year', y='life_exp', color='country', markers=True,
    title='11. Life Expectancy Trend for Selected Countries (1998–2007)',
    labels={'life_exp': 'Life Expectancy (years)', 'year': 'Year'},
    template='plotly_white')

fig_co2_trend = px.line(co2_trend, x='year', y='co2_consump', color='continent', markers=True,
    title='12. Avg CO₂ Consumption by Continent Over Time',
    labels={'co2_consump': 'CO₂ per Capita (metric tons)', 'year': 'Year'},
    template='plotly_white')

fig_hdi_hist = px.histogram(df_2007_dash, x='hdi_index', color='continent', nbins=30,
    title='13. HDI Index Distribution (2007)',
    labels={'hdi_index': 'Human Development Index'},
    template='plotly_white', barmode='overlay', opacity=0.75)

fig_top10 = px.bar(top10, x='gdp', y='country', color='continent', orientation='h',
    title='14. Top 10 Countries by GDP per Capita (2007)',
    labels={'gdp': 'GDP per Capita (USD)', 'country': 'Country'},
    template='plotly_white', text_auto='.0f')

fig_gdp_time = px.bar(gdp_cont_year, x='year', y='gdp', color='continent', barmode='group',
    title='15. Avg GDP per Capita by Continent Over Time',
    labels={'gdp': 'Avg GDP per Capita (USD)', 'year': 'Year'},
    template='plotly_white')

# ── Build Dash App ───────────────────────────────────────────────
app = dash.Dash(__name__)

card_style = {'width': '49%', 'display': 'inline-block', 'padding': '10px'}
full_style = {'width': '99%', 'display': 'inline-block', 'padding': '10px'}

app.layout = html.Div([

    # Header
    html.Div([
        html.H1('🌍 Global Development Dashboard',
                style={'textAlign': 'center', 'color': '#2c3e50', 'fontFamily': 'Arial'}),
        html.P('Gapminder Dataset — All 15 Visualizations | Plotly & Dash',
               style={'textAlign': 'center', 'color': '#7f8c8d', 'fontFamily': 'Arial'})
    ], style={'padding': '20px 0 10px 0', 'borderBottom': '2px solid #ecf0f1'}),

    # Row 1
    html.Div([
        html.Div([dcc.Graph(figure=fig_hist)],    style=card_style),
        html.Div([dcc.Graph(figure=fig_line)],    style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 2
    html.Div([
        html.Div([dcc.Graph(figure=fig_scatter)],  style=card_style),
        html.Div([dcc.Graph(figure=fig_bar_top20)], style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 3 — Full width map
    html.Div([
        html.Div([dcc.Graph(figure=fig_map)], style=full_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 4
    html.Div([
        html.Div([dcc.Graph(figure=fig_box_life)], style=card_style),
        html.Div([dcc.Graph(figure=fig_box_gdp)],  style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 5
    html.Div([
        html.Div([dcc.Graph(figure=fig_co2_gdp)], style=card_style),
        html.Div([dcc.Graph(figure=fig_hdi)],      style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 6
    html.Div([
        html.Div([dcc.Graph(figure=fig_services)],      style=card_style),
        html.Div([dcc.Graph(figure=fig_country_trend)], style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 7
    html.Div([
        html.Div([dcc.Graph(figure=fig_co2_trend)], style=card_style),
        html.Div([dcc.Graph(figure=fig_hdi_hist)],  style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Row 8
    html.Div([
        html.Div([dcc.Graph(figure=fig_top10)],    style=card_style),
        html.Div([dcc.Graph(figure=fig_gdp_time)], style=card_style),
    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

    # Footer
    html.Div(
        html.P('Data Source: Gapminder Foundation | Built with Plotly Dash',
               style={'textAlign': 'center', 'color': '#95a5a6', 'fontFamily': 'Arial'}),
        style={'borderTop': '1px solid #ecf0f1', 'padding': '15px 0'}
    )

], style={'backgroundColor': '#f9f9f9', 'minHeight': '100vh'})

print("Dashboard running at: http://127.0.0.1:8050")
app.run(debug=False, port=8050, jupyter_mode="external")

Dashboard running at: http://127.0.0.1:8050
Dash app running on http://127.0.0.1:8050/


---

## Conclusion

**GDP and Life Expectancy**
- There is a strong positive correlation between GDP per capita and life expectancy
- Countries with higher GDP consistently show longer life expectancy
- However, the relationship is non-linear — gains diminish at very high GDP levels

**Differences Between Continents**
- Europe and North America lead in GDP, HDI, and life expectancy
- Africa has the lowest life expectancy and HDI scores across all years
- Asia shows the widest internal variation — from very poor to very wealthy nations
- South America sits in the middle tier across most indicators

**CO₂ and Development**
- Wealthier nations (especially North America and Europe) produce far more CO₂ per capita
- Asia's CO₂ is rising as its economies develop rapidly
- Africa has the lowest CO₂ consumption, reflecting lower industrialization

**Global Trends (1998–2007)**
- Life expectancy improved across all continents during this period
- GDP grew steadily in most regions, with Asia showing the strongest growth
- HDI improved globally, indicating progress in education and health worldwide


---

In [25]:
# Save cleaned data to CSV
df.to_csv('gapminder_cleaned.csv', index=False)
print("Cleaned data saved as: gapminder_cleaned.csv ✅")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

Cleaned data saved as: gapminder_cleaned.csv ✅
Shape: (3675, 8)
Missing values: 0
